https://www.kaggle.com/competitions/nlp-getting-started/code

In [1]:
!pip install optuna

In [2]:
!pip install keybert

In [3]:
!pip install rake-nltk

In [4]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, TargetEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from keybert import KeyBERT
from sklearn.model_selection import train_test_split, cross_val_score
import lightgbm as lgb
from sklearn.metrics import f1_score
import keras_nlp
import tensorflow as tf
from tensorflow import keras
from rake_nltk import Rake
from nltk.corpus import stopwords
import nltk
import re
import string
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from sklearn.ensemble import VotingClassifier
import optuna
from optuna.pruners import MedianPruner
from sklearn.linear_model import LogisticRegression

In [5]:
nltk.download('stopwords')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [6]:
import tensorflow_text
from google.colab import drive

In [7]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
df_train = pd.read_csv('/content/drive/MyDrive/ML_junior/Проекты/NLP/train.csv')

In [9]:
df_test = pd.read_csv('/content/drive/MyDrive/ML_junior/Проекты/NLP/test.csv')

In [91]:
df_test

,id,keyword,location,text,text_lengt,word_count,count_hesh,count_email,url_count,caps_count,caps_ratio,exclamation
0,0,NaN,NaN,just happened a terrible car crash,34,6,0,0,0,1,0.028571,0
1,2,NaN,NaN,"heard about earthquake is different cities, st...",64,9,1,0,0,1,0.015385,0
2,3,NaN,NaN,"there is a forest fire at spot pond, geese are...",96,19,0,0,0,1,0.010309,0
3,9,NaN,NaN,apocalypse lighting. spokane wildfires,40,4,2,0,0,2,0.048780,0
4,11,NaN,NaN,typhoon soudelor kills 28 in china and taiwan,45,8,0,0,0,4,0.086957,0
...,...,...,...,...,...,...,...,...,...,...,...,...
3258,10861,NaN,NaN,earthquake safety los angeles ûò safety faste...,55,8,0,0,0,45,0.803571,0
3259,10865,NaN,NaN,storm in ri worse than last hurricane. my city...,139,23,0,0,0,7,0.050000,0
3260,10868,NaN,NaN,green line derailment in chicago,55,6,0,0,1,9,0.160714,0
3261,10874,NaN,NaN,meg issues hazardous weather outlook (hwo),65,7,0,0,1,15,0.227273,0


In [10]:
df_train.head()

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1


In [11]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7613 entries, 0 to 7612
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id        7613 non-null   int64 
 1   keyword   7552 non-null   object
 2   location  5080 non-null   object
 3   text      7613 non-null   object
 4   target    7613 non-null   int64 
dtypes: int64(2), object(3)
memory usage: 297.5+ KB


In [12]:
df_train['target'].value_counts()

,count
target,
0,4342
1,3271


In [13]:
df_train['keyword'].unique()

array([nan, 'ablaze', 'accident', 'aftershock', 'airplane%20accident',
       'ambulance', 'annihilated', 'annihilation', 'apocalypse',
       'armageddon', 'army', 'arson', 'arsonist', 'attack', 'attacked',
       'avalanche', 'battle', 'bioterror', 'bioterrorism', 'blaze',
       'blazing', 'bleeding', 'blew%20up', 'blight', 'blizzard', 'blood',
       'bloody', 'blown%20up', 'body%20bag', 'body%20bagging',
       'body%20bags', 'bomb', 'bombed', 'bombing', 'bridge%20collapse',
       'buildings%20burning', 'buildings%20on%20fire', 'burned',
       'burning', 'burning%20buildings', 'bush%20fires', 'casualties',
       'casualty', 'catastrophe', 'catastrophic', 'chemical%20emergency',
       'cliff%20fall', 'collapse', 'collapsed', 'collide', 'collided',
       'collision', 'crash', 'crashed', 'crush', 'crushed', 'curfew',
       'cyclone', 'damage', 'danger', 'dead', 'death', 'deaths', 'debris',
       'deluge', 'deluged', 'demolish', 'demolished', 'demolition',
       'derail', 'der

In [14]:
df_train['location'].unique()

array([nan, 'Birmingham', 'Est. September 2012 - Bristol', ...,
       'Vancouver, Canada', 'London ', 'Lincoln'], dtype=object)

In [15]:
#kw_model = KeyBERT()

In [16]:
#def extract_keywords_bert(text):
    #keywords = kw_model.extract_keywords(text, keyphrase_ngram_range = (1,2), stop_words = 'english', top_n=5)
    #return [kw[0] for kw in keywords]

In [17]:
#df_train['keyword_new'] = df_train['text'].apply(extract_keywords_bert)

In [18]:
def extrakt_rake(text, n = 3):
  rake = Rake(stopwords = stopwords.words('english'))
  rake.extract_keywords_from_text(text)
  keywords = rake.get_ranked_phrases()[:n]
  return ' '.join(keywords)



In [19]:
mask = df_train['keyword'].isna()
df_train.loc[mask,'keyword'] = df_train.loc[mask,'text'].apply(extrakt_rake, n = 3)

In [20]:
mode_value = df_train['location'].mode()[0]
df_train['location'] = df_train['location'].fillna(mode_value)

In [21]:
df_train['text_lengt']  = df_train['text'].str.len()
df_test['text_lengt'] = df_test['text'].str.len()
df_train['word_count']  = df_train['text'].str.split().str.len()
df_test['word_count'] = df_test['text'].str.split().str.len()

In [22]:
df_train['count_hesh'] = df_train['text'].str.count('#')
df_train['count_email'] = df_train['text'].str.count('@')
df_train['url_count'] = df_train['text'].str.count('http')

df_test['count_hesh'] = df_test['text'].str.count('#')
df_test['count_email'] = df_test['text'].str.count('@')
df_test['url_count'] = df_test['text'].str.count('http')


In [23]:
df_train['caps_count'] = df_train['text'].apply(lambda x: sum(1 for c in x if c.isupper()))
df_train['caps_ratio'] = df_train['caps_count'] / (df_train['text_lengt']+1)

df_test['caps_count'] = df_test['text'].apply(lambda x: sum(1 for c in x if c.isupper()))
df_test['caps_ratio'] = df_test['caps_count'] / (df_test['text_lengt']+1)

In [24]:
df_train['exclamation'] = df_train['text'].str.count('!')
df_test['exclamation'] = df_test['text'].str.count('!')

In [25]:
def clean_text(text):
  text = text.lower()
  text = re.sub(r'@\w+','',text)
  text = re.sub(r'#(\w+)',r'\1',text)
  text = re.sub(r'http\S+|www\S+|https\S+', '', text)
  text = re.sub(r'\s+',' ',text).strip()
  return text

df_train['text'] = df_train['text'].apply(clean_text)
df_test['text'] = df_test['text'].apply(clean_text)

In [26]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7613 entries, 0 to 7612
Data columns (total 13 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   id           7613 non-null   int64  
 1   keyword      7613 non-null   object 
 2   location     7613 non-null   object 
 3   text         7613 non-null   object 
 4   target       7613 non-null   int64  
 5   text_lengt   7613 non-null   int64  
 6   word_count   7613 non-null   int64  
 7   count_hesh   7613 non-null   int64  
 8   count_email  7613 non-null   int64  
 9   url_count    7613 non-null   int64  
 10  caps_count   7613 non-null   int64  
 11  caps_ratio   7613 non-null   float64
 12  exclamation  7613 non-null   int64  
dtypes: float64(1), int64(9), object(3)
memory usage: 773.3+ KB


In [27]:
X = df_train.drop(['id','target'], axis = 1)
y = df_train['target']
X_train_full, X_test_final, y_train_full, y_test_final = train_test_split(X, y, test_size = 0.2, random_state = 42, stratify = y)


In [28]:
target_encoder  = TargetEncoder(smooth = 'auto', random_state = 42)

In [29]:
def for_astype(x):
  for col in ['keyword','location']:
    x[col] = x[col].astype(str)
  return x
for_astype(X_train_full)
for_astype(X_test_final)

,keyword,location,text,text_lengt,word_count,count_hesh,count_email,url_count,caps_count,caps_ratio,exclamation
4863,mass%20murderer,"Huntsville, AL",step one: get that mass murderer's portrait of...,71,11,0,1,0,3,0.041667,0
1370,bush%20fires,Queen Creek AZ,ted cruz fires back at jeb &amp; bush: ûïwe l...,130,20,0,0,1,19,0.145038,0
3521,eyewitness,USA,how ûïlittle boyû affected the people in hi...,104,12,0,0,1,19,0.180952,0
178,ambulance,Happily Married with 2 kids,ambulance sprinter automatic frontline vehicle...,103,13,0,0,1,67,0.644231,0
5859,ruin,USA,'cause you play me like a symphony play me til...,112,20,0,0,0,3,0.026549,0
...,...,...,...,...,...,...,...,...,...,...,...
6939,trouble,USA,frickin summer and its humidity building up an...,80,11,0,1,0,2,0.024691,0
2074,dead,dundalk ireland,is ross really dead?? askcharley,44,6,1,1,0,3,0.066667,0
3186,emergency%20plan,"Atlanta, GA",b/c it costs less to have sick people using em...,95,12,0,0,1,4,0.041667,0
4297,hellfire,USA,then i do this to one of them ????,58,11,0,2,0,9,0.152542,0


In [30]:
X_train_enc = X_train_full.copy()
X_temp_enc = X_test_final.copy()

In [31]:
cat_col = ['keyword','location']
X_train_enc[cat_col] = target_encoder.fit_transform(X_train_enc[cat_col],y_train_full)
X_temp_enc[cat_col] = target_encoder.transform(X_temp_enc[cat_col])

In [32]:
tfidf = TfidfVectorizer(min_df=3,max_df=0.8, stop_words='english', ngram_range=(1,3), max_features = 15000,
    sublinear_tf = True,
    analyzer='word',
    strip_accents='ascii',
    lowercase=True,
    token_pattern=r'(?u)\b\w+\b')

In [33]:
vectoriz_train = tfidf.fit_transform(X_train_enc['text'])
vectoriz_temp = tfidf.transform(X_temp_enc['text'])


In [34]:
vectoriz_df = pd.DataFrame(vectoriz_train.toarray(), columns = [f'tfidf_{word}' for word in tfidf.get_feature_names_out()], index = X_train_enc.index)
vectoriz_df_test = pd.DataFrame(vectoriz_temp.toarray(), columns = [f'tfidf_{word}' for word in tfidf.get_feature_names_out()], index = X_temp_enc.index)


In [35]:
X_train_enc = X_train_enc.drop('text', axis = 1)
X_temp_enc = X_temp_enc.drop('text', axis = 1)
X_train_final = pd.concat([X_train_enc, vectoriz_df], axis = 1)
X_temp_final = pd.concat([X_temp_enc, vectoriz_df_test], axis = 1)

In [36]:
study = optuna.create_study(sampler = optuna.samplers.TPESampler(seed=42, n_startup_trials=5,
n_ei_candidates = 64,multivariate=True),pruner = MedianPruner(
    n_startup_trials=10,
    n_warmup_steps=20,
    interval_steps=10
), study_name = 'my_first_study', direction = 'maximize')

/tmp/ipykernel_49363/982616621.py:1: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  study = optuna.create_study(sampler = optuna.samplers.TPESampler(seed=42, n_startup_trials=5,
[I 2026-08-21 16:21:27,536] A new study created in memory with name: my_first_study


In [37]:
def objective(trial):
    estimator = trial.suggest_int('n_estimators',300,500)
    learning_rate = trial.suggest_float('learning_rate',0.02,0.04)
    num_leaves = trial.suggest_int('num_leaves', 19,39)
    max_depth = trial.suggest_int('max_depth', 5,8)
    min_child_samples  = trial.suggest_int('min_child_samples', 15, 40)
    min_split_gain = trial.suggest_float('min_split_gain',0.01,0.1)
    subsample = trial.suggest_float('subsample', 0.7,1)
    colsample_bytree = trial.suggest_float('colsample_bytree',0.6,0.9)
    subsample_freq = trial.suggest_int('subsample_freq', 5,10)

    model  = lgb.LGBMClassifier(
    n_estimators=estimator,
    learning_rate=learning_rate,
    num_leaves=num_leaves,
    max_depth=max_depth,
    class_weight='balanced',
    min_child_samples=min_child_samples,
    min_split_gain=min_split_gain,
    subsample=subsample,
    colsample_bytree=colsample_bytree,
    subsample_freq=subsample_freq,

    n_jobs=-1,
    random_state=42,
    verbose=-1)
    scores = cross_val_score(model, X_train_final, y_train_full,cv=3,scoring ='f1', n_jobs = 1)
    return scores.mean()

In [38]:
study.optimize(objective,n_trials = 50, show_progress_bar = True)

  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-08-21 16:21:47,191] Trial 0 finished with value: 0.7114243145854576 and parameters: {'n_estimators': 375, 'learning_rate': 0.03901428612819832, 'num_leaves': 34, 'max_depth': 7, 'min_child_samples': 19, 'min_split_gain': 0.02403950683025824, 'subsample': 0.7174250836504598, 'colsample_bytree': 0.8598528437324806, 'subsample_freq': 8}. Best is trial 0 with value: 0.7114243145854576.
[I 2026-08-21 16:22:07,960] Trial 1 finished with value: 0.7146609020556637 and parameters: {'n_estimators': 442, 'learning_rate': 0.02041168988591605, 'num_leaves': 39, 'max_depth': 8, 'min_child_samples': 20, 'min_split_gain': 0.02636424704863906, 'subsample': 0.7550213529560301, 'colsample_bytree': 0.6912726728878613, 'subsample_freq': 8}. Best is trial 1 with value: 0.7146609020556637.
[I 2026-08-21 16:22:16,833] Trial 2 finished with value: 0.7183932985353808 and parameters: {'n_estimators': 386, 'learning_rate': 0.025824582803960838, 'num_leaves': 31, 'max_depth': 5, 'min_child_samples': 22, 'm

In [39]:
best_params = study.best_params

In [40]:
lgb_model = lgb.LGBMClassifier(**best_params)

In [41]:
lgb_model.fit(X_train_final, y_train_full)

LGBMClassifier(colsample_bytree=0.6093396102780098,
               learning_rate=0.032403193460540984, max_depth=6,
               min_child_samples=15, min_split_gain=0.0632781882265206,
               n_estimators=482, num_leaves=28, subsample=0.9153956633619522,
               subsample_freq=5)

In [42]:
predict_train = lgb_model.predict(X_train_final)
predict_test = lgb_model.predict(X_temp_final)

In [43]:
print(f'lgbm train {f1_score(y_train_full,predict_train)}')
print(f'lgbm test {f1_score(y_test_final,predict_test)}')

lgbm train 0.7906408952187182
lgbm test 0.7532258064516129


keybert
train 0.7555850475558504

test 0.7337278106508875


rake
lgbm train 0.8077071290944123

lgbm test 0.7561349693251533

In [44]:
xgb_model = XGBClassifier(n_estimators=300,learning_rate=0.03,max_depth=4,gamma=0.1,
    subsample=0.7,
    colsample_bytree=0.7,
    colsample_bylevel=0.7,
    random_state=42,
    objective='binary:logistic',
    eval_metric='logloss',
    n_jobs=-1,
    verbosity=0
)

In [45]:
xgb_model.fit(X_train_final, y_train_full)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=0.7, colsample_bynode=None,
              colsample_bytree=0.7, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=0.1,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.03, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=4, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=300, n_jobs=-1,
              num_parallel_tree=None, ...)

In [46]:
predict_train_xgb = xgb_model.predict(X_train_final)
predict_test_xgb = xgb_model.predict(X_temp_final)

In [47]:
print(f'ансамбль train {f1_score(y_train_full,predict_train_xgb)}')
print(f'ансамбль test {f1_score(y_test_final,predict_test_xgb)}')

ансамбль train 0.7437650844730491
ансамбль test 0.7414634146341463


ансамбль train 0.7437650844730491

ансамбль test 0.7414634146341463

In [48]:
estimator = [('lgb', lgb.LGBMClassifier(**best_params)),
                      ('xgb',  xgb_model)]
ensemble = VotingClassifier(estimators = estimator,voting = 'soft')

In [49]:
ensemble.fit(X_train_final, y_train_full)

VotingClassifier(estimators=[('lgb',
                              LGBMClassifier(colsample_bytree=0.6093396102780098,
                                             learning_rate=0.032403193460540984,
                                             max_depth=6, min_child_samples=15,
                                             min_split_gain=0.0632781882265206,
                                             n_estimators=482, num_leaves=28,
                                             subsample=0.9153956633619522,
                                             subsample_freq=5)),
                             ('xgb',
                              XGBClassifier(base_score=None, booster=None,
                                            callbacks=None,
                                            colsample_bylevel...
                                            feature_weights=None, gamma=0.1,
                                            grow_policy=None,
                                            importance_type=None,
                                            interaction_constraints=None,
                                            learning_rate=0.03, max_bin=None,
                                            max_cat_threshold=None,
                                            max_cat_to_onehot=None,
                                            max_delta_step=None, max_depth=4,
                                            max_leaves=None,
                                            min_child_weight=None, missing=nan,
                                            monotone_constraints=None,
                                            multi_strategy=None,
                                            n_estimators=300, n_jobs=-1,
                                            num_parallel_tree=None, ...))],
                 voting='soft')

In [50]:
ensemble_pred_train = ensemble.predict(X_train_final)
ensemble_pred_test = ensemble.predict(X_temp_final)

In [51]:
print(f'ensemble train {f1_score(y_train_full,ensemble_pred_train)}')
print(f'ensemble test {f1_score(y_test_final,ensemble_pred_test)}')

ensemble train 0.7703823588913615
ensemble test 0.752827140549273


lgbm train 0.8284960422163589

lgbm test 0.7550525464834277

In [52]:
###|BERTA

In [53]:
preset = 'distil_bert_base_en_uncased'

In [54]:
preprocessor = keras_nlp.models.DistilBertPreprocessor.from_preset(preset, sequence_length=16, name = 'preproc_tweet_1')

In [55]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train_full),
    y=y_train_full
)
class_weight_dict = dict(enumerate(class_weights))

In [56]:
classifier = keras_nlp.models.DistilBertClassifier.from_preset(preset,preprocessor = preprocessor,num_classes = 2, dropout = 0.3)

In [57]:
classifier.compile(loss= keras.losses.SparseCategoricalCrossentropy(from_logits = True),
                   optimizer=keras.optimizers.Adam(learning_rate=2e-5),
                   metrics=['accuracy'])

In [58]:
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size = 0.3, random_state = 42, stratify = y_train_full)

In [59]:
history = classifier.fit(X_train['text'], y_train, batch_size = 32,epochs=2,class_weight = class_weight_dict ,validation_data=(X_val['text'],y_val))

Epoch 1/2
134/134 ━━━━━━━━━━━━━━━━━━━━ 2199s 16s/step - accuracy: 0.7535 - loss: 0.5293 - val_accuracy: 0.8183 - val_loss: 0.4224
Epoch 2/2
134/134 ━━━━━━━━━━━━━━━━━━━━ 2137s 16s/step - accuracy: 0.8297 - loss: 0.4168 - val_accuracy: 0.8227 - val_loss: 0.4091


In [61]:
y_pred_train = classifier.predict(X_train_full['text'])

191/191 ━━━━━━━━━━━━━━━━━━━━ 665s 3s/step


In [62]:
y_pred_test = classifier.predict(X_test_final['text'])

48/48 ━━━━━━━━━━━━━━━━━━━━ 161s 3s/step


In [63]:
train_class = np.argmax(y_pred_train,axis=1)
test_class = np.argmax(y_pred_test,axis = 1)
print(f'train {f1_score(y_train_full ,train_class)}')
print(f'test {f1_score(y_test_final,test_class)}')

train 0.8299570288520565
test 0.7957860615883307


train 0.7722174288179465

test 0.7476149176062445

In [64]:
predict_final_test = classifier.predict(df_test['text'])

102/102 ━━━━━━━━━━━━━━━━━━━━ 342s 3s/step


In [65]:
final_class = np.argmax(predict_final_test, axis = 1)

In [66]:
sub = pd.DataFrame({'id': df_test['id'], 'target': final_class})

In [67]:
save_path = '/content/drive/MyDrive/ML_junior/Проекты/NLP/submission.csv'

In [68]:
sub.to_csv(save_path, index = False)

In [69]:
##ансамбль логрег, берты и гб

In [71]:
lgb_prob_train = lgb_model.predict_proba(X_train_final)[:,1]
lgb_prob_test = lgb_model.predict_proba(X_temp_final)[:,1]

In [ ]:
#bert_probs_train = tf.nn.softmax(y_pred_train).numpy()[:,1]
#bert_probs_test = tf.nn.softmax(y_pred_test).numpy()[:,1]

In [77]:
bert_prob_train = tf.nn.softmax(y_pred_train).numpy()[:, 1]
bert_prob_test = tf.nn.softmax(y_pred_test).numpy()[:, 1]

In [78]:
meta_features_train = np.column_stack([lgb_prob_train, bert_prob_train])
meta_featurs_test = np.column_stack([lgb_prob_test, bert_prob_test])

In [79]:
meta_model = LogisticRegression(max_iter=1000, C=1.0, random_state=42)

In [80]:
meta_model.fit(meta_features_train, y_train_full)

LogisticRegression(max_iter=1000, random_state=42)

In [83]:
meta_pred_train = meta_model.predict(meta_features_train)
meta_pred_test = meta_model.predict(meta_featurs_test)

In [85]:
print(f'train {f1_score(y_train_full ,meta_pred_train)}')
print(f'train {f1_score(y_test_final ,meta_pred_test)}')

train 0.8492985971943888
train 0.8066298342541437
